# 🔗 Notebook 3 — Projet Intégrateur : Pipeline Hybride ML + DL

**Master 1 Intelligence Artificielle — Université de Nouakchott (UN-FST) — 2026**
Groupe : Mohamed Salem Ebnou Oubeid (C34613) · Fatimata Issa Saw (C21304) · Oussama Sid'Ahmed Hedy (C34603)

---

Ce notebook implémente le **pipeline hybride** présenté dans la partie Projet :

```
[Images MNIST]
      ↓
[MLP extracteur — features de la couche cachée]
      ↓
[PCA — réduction dimensionnelle]
      ↓
[XGBoost — méta-classifieur d'ensemble]
      ↓
[Prédiction finale]
```

L'idée clé : le MLP apprend des **représentations riches** des images,
et XGBoost exploite ces représentations pour une classification robuste.

## Table des matières
1. Imports et configuration
2. Chargement des données (MNIST ou Digits)
3. Exploration des données
4. MLP extracteur de features
5. Extraction des features de la couche cachée
6. Réduction dimensionnelle — PCA
7. Classification — XGBoost sur les features réduites
8. Comparaison globale des pipelines
9. Interprétabilité — SHAP values
10. Incertitude — Prédictions avec variance (Bootstrap)
11. Analyse par classe
12. Conclusion


## 1. Imports et configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              ConfusionMatrixDisplay, confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("✅ XGBoost disponible")
except ImportError:
    HAS_XGB = False
    print("⚠️  XGBoost non disponible — pip install xgboost")

# SHAP
try:
    import shap
    HAS_SHAP = True
    print("✅ SHAP disponible")
except ImportError:
    HAS_SHAP = False
    print("⚠️  SHAP non disponible — pip install shap")

# Couleurs UN-FST
C_GREEN  = '#3A8E5B'
C_GOLD   = '#E8B608'
C_BLUE   = '#1ea8c4'
C_RED    = '#e74c3c'

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print("✅ Imports OK")


## 2. Chargement des données

Nous essayons d'abord de charger **MNIST complet** (784 features, 70 000 images)
via `fetch_openml`. Si le téléchargement échoue, on utilise le dataset **Digits**
(64 features, 1 797 images) comme fallback.


In [ ]:
# ── Chargement MNIST ou Digits ──
try:
    from sklearn.datasets import fetch_openml
    print("📥 Chargement de MNIST depuis OpenML (peut prendre 30–60 s)...")
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X_raw = mnist.data.astype(np.float32) / 255.0
    y_raw = mnist.target.astype(np.int32)

    # Sous-ensemble de 12 000 images pour la rapidité de la démo
    X_raw, _, y_raw, _ = train_test_split(
        X_raw, y_raw, train_size=12000, random_state=42, stratify=y_raw
    )
    N_FEATURES   = 784
    IMG_SHAPE    = (28, 28)
    DATASET_NAME = "MNIST (28×28)"
    print(f"✅ MNIST chargé : {X_raw.shape}")

except Exception as e:
    print(f"⚠️  Fallback sur Digits (raison : {type(e).__name__})")
    data  = load_digits()
    X_raw = data.data.astype(np.float32) / 16.0
    y_raw = data.target.astype(np.int32)
    N_FEATURES   = 64
    IMG_SHAPE    = (8, 8)
    DATASET_NAME = "Digits (8×8)"
    print(f"✅ Digits chargé : {X_raw.shape}")

# ── Division train / validation / test (60 % / 20 % / 20 %) ──
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp
)

print(f"\nDataset    : {DATASET_NAME}  — {N_FEATURES} features, 10 classes")
print(f"Train      : {X_train.shape[0]}  | Val : {X_val.shape[0]}  | Test : {X_test.shape[0]}")


## 3. Exploration des données

In [ ]:
# Distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogramme de la distribution
counts = np.bincount(y_train)
axes[0].bar(range(10), counts, color=C_GREEN, edgecolor='white')
axes[0].set_xlabel("Chiffre"); axes[0].set_ylabel("Nombre d'exemples")
axes[0].set_title("Distribution des classes (entraînement)", fontweight='bold')
axes[0].set_xticks(range(10))

# Visualisation de 20 images
n_row, n_col = 2, 10
fig2, axes2 = plt.subplots(n_row, n_col, figsize=(14, 3))
for label_show in range(n_col):
    idx = np.where(y_train == label_show)[0][0]
    for row in range(n_row):
        idx2 = np.where(y_train == label_show)[0][row]
        axes2[row, label_show].imshow(
            X_train[idx2].reshape(IMG_SHAPE), cmap='gray_r'
        )
        axes2[row, label_show].set_title(str(label_show), fontsize=8)
        axes2[row, label_show].axis('off')

plt.suptitle(f"Exemples — {DATASET_NAME}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Variance moyenne des pixels : {X_train.var(axis=0).mean():.4f}")
print(f"Pixels quasi constants (var < 0.001) : {(X_train.var(axis=0) < 0.001).sum()}")


## 4. MLP extracteur de features

Nous entraînons un MLP complet sur les données brutes, puis nous récupérons
les activations de sa **dernière couche cachée** comme nouvelles features.

Cette approche s'appelle **transfer learning** ou **feature extraction** :
le MLP apprend une représentation comprimée et informative des images.


In [ ]:
# ── Entraînement du MLP extracteur ──
hidden_size = 128  # taille de la dernière couche cachée (nos futures features)

mlp_feat = MLPClassifier(
    hidden_layer_sizes=(256, hidden_size),   # 2 couches cachées : 256 → 128
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=30,
    random_state=42,
    verbose=False,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5
)

print("Entraînement du MLP extracteur...")
t0 = time.time()
mlp_feat.fit(X_train, y_train)
temps_mlp = time.time() - t0

acc_mlp = accuracy_score(y_test, mlp_feat.predict(X_test))
print(f"✅ MLP end-to-end — Accuracy test : {acc_mlp:.4f}  ({temps_mlp:.1f}s)")
print(f"   Itérations convergence : {mlp_feat.n_iter_}")


## 5. Extraction des features de la couche cachée

Sklearn ne fournit pas directement les activations cachées, mais nous pouvons
les calculer manuellement en appliquant la transformation couche par couche.


In [ ]:
def extraire_features(mlp, X):
    """
    Extrait les activations de la dernière couche cachée d'un MLPClassifier sklearn.

    Paramètres
    ----------
    mlp : MLPClassifier sklearn entraîné
    X   : array (n_samples, n_features) — données d'entrée

    Retourne
    --------
    activations : array (n_samples, n_hidden_last) — features extraites
    """
    activations = X.copy()

    # Parcourt toutes les couches cachées (pas la couche de sortie)
    for i, (W, b) in enumerate(zip(mlp.coefs_[:-1], mlp.intercepts_[:-1])):
        z           = activations @ W + b
        activations = np.maximum(0, z)   # activation ReLU

    return activations


# Extraction des features sur train, val, test
print("Extraction des features cachées...")
F_train = extraire_features(mlp_feat, X_train)
F_val   = extraire_features(mlp_feat, X_val)
F_test  = extraire_features(mlp_feat, X_test)

print(f"Features extraites : {F_train.shape}  (n_samples × n_hidden_last)")
print(f"  → Réduction : {N_FEATURES} pixels → {F_train.shape[1]} features apprises")

# Visualisation des features (t-SNE si temps disponible, sinon PCA 2D)
pca_visu = PCA(n_components=2, random_state=42)
F_2d     = pca_visu.fit_transform(F_test)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(F_2d[:, 0], F_2d[:, 1], c=y_test,
                      cmap='tab10', s=12, alpha=0.7)
plt.colorbar(scatter, label="Chiffre")
plt.title("Features MLP — projection PCA 2D (test)", fontweight='bold')
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.tight_layout()
plt.show()


## 6. Réduction dimensionnelle — PCA

L'**Analyse en Composantes Principales** (PCA) réduit la dimensionnalité des features
tout en conservant le maximum de variance.

Avantages :
- Réduit le risque de *malédiction de la dimension* pour XGBoost
- Accélère l'entraînement du méta-classifieur
- Peut améliorer la généralisation


In [ ]:
# Variance expliquée en fonction du nombre de composantes
pca_full = PCA(random_state=42).fit(F_train)
var_cumulee = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].plot(var_cumulee, color=C_GREEN, lw=2)
axes[0].axhline(0.90, color=C_RED,  ls='--', lw=1.5, label='90% variance')
axes[0].axhline(0.95, color=C_GOLD, ls='--', lw=1.5, label='95% variance')
axes[0].set_xlabel("Nombre de composantes principales")
axes[0].set_ylabel("Variance expliquée cumulée")
axes[0].set_title("Scree plot — PCA sur features MLP", fontweight='bold')
axes[0].legend()

# Nombre de composantes nécessaires
n_90 = np.searchsorted(var_cumulee, 0.90) + 1
n_95 = np.searchsorted(var_cumulee, 0.95) + 1
axes[0].axvline(n_90, color=C_RED,  ls=':', alpha=0.6)
axes[0].axvline(n_95, color=C_GOLD, ls=':', alpha=0.6)

# Choix de n_components
N_PCA = min(50, F_train.shape[1])
pca   = PCA(n_components=N_PCA, random_state=42)
P_train = pca.fit_transform(F_train)
P_val   = pca.transform(F_val)
P_test  = pca.transform(F_test)

axes[1].bar(range(1, min(21, N_PCA+1)),
            pca.explained_variance_ratio_[:20],
            color=C_BLUE, edgecolor='white')
axes[1].set_xlabel("Composante principale")
axes[1].set_ylabel("Variance expliquée")
axes[1].set_title(f"Variance par composante (top 20 / {N_PCA} retenues)",
                  fontweight='bold')

plt.tight_layout()
plt.show()

var_retenue = pca.explained_variance_ratio_.sum()
print(f"Composantes retenues : {N_PCA}")
print(f"Variance expliquée   : {var_retenue:.2%}")
print(f"Dimensions : {F_train.shape[1]} → {P_train.shape[1]}")
print(f"90% variance en {n_90} composantes | 95% en {n_95} composantes")


## 7. Classification — XGBoost sur les features réduites

XGBoost est entraîné sur les features issues de MLP + PCA.
C'est le **méta-classifieur** du pipeline hybride.


In [ ]:
resultats = {}
temps_entrainement = {}

# ── Pipeline hybride : MLP features → PCA → XGBoost ──
if HAS_XGB:
    t0  = time.time()
    xgb = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    xgb.fit(P_train, y_train,
            eval_set=[(P_val, y_val)],
            verbose=False)
    t1 = time.time()

    acc_hybride = accuracy_score(y_test, xgb.predict(P_test))
    resultats['Pipeline hybride (MLP+PCA+XGB)'] = acc_hybride
    temps_entrainement['Pipeline hybride (MLP+PCA+XGB)'] = round(t1 - t0, 2)
    print(f"✅ Pipeline hybride MLP→PCA→XGBoost : {acc_hybride:.4f}")
else:
    # Fallback : Random Forest
    t0  = time.time()
    rf  = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(P_train, y_train)
    acc_hybride = accuracy_score(y_test, rf.predict(P_test))
    resultats['Pipeline hybride (MLP+PCA+RF)'] = acc_hybride
    temps_entrainement['Pipeline hybride (MLP+PCA+RF)'] = round(time.time()-t0, 2)
    print(f"✅ Pipeline hybride MLP→PCA→RF : {acc_hybride:.4f}")


## 8. Comparaison globale des pipelines

In [ ]:
# ── Baselines à comparer ──

# 1. Régression logistique sur pixels bruts
t0  = time.time()
lr  = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)
resultats['LR (pixels bruts)'] = accuracy_score(y_test, lr.predict(X_test))
temps_entrainement['LR (pixels bruts)'] = round(time.time() - t0, 2)

# 2. Random Forest sur pixels bruts
t0  = time.time()
rf_brut = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_brut.fit(X_train, y_train)
resultats['RF (pixels bruts)'] = accuracy_score(y_test, rf_brut.predict(X_test))
temps_entrainement['RF (pixels bruts)'] = round(time.time() - t0, 2)

# 3. XGBoost sur pixels bruts (si disponible)
if HAS_XGB:
    t0 = time.time()
    xgb_brut = XGBClassifier(
        n_estimators=100, eval_metric='mlogloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_brut.fit(X_train, y_train)
    resultats['XGB (pixels bruts)'] = accuracy_score(y_test, xgb_brut.predict(X_test))
    temps_entrainement['XGB (pixels bruts)'] = round(time.time() - t0, 2)

# 4. MLP end-to-end (déjà calculé)
resultats['MLP end-to-end']  = acc_mlp
temps_entrainement['MLP end-to-end'] = round(temps_mlp, 2)

# 5. RF sur features MLP (sans PCA)
t0  = time.time()
rf_feat = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_feat.fit(F_train, y_train)
resultats['RF (features MLP)'] = accuracy_score(y_test, rf_feat.predict(F_test))
temps_entrainement['RF (features MLP)'] = round(time.time() - t0, 2)

# ── Tableau récapitulatif ──
print(f"{'Pipeline':<40} {'Accuracy':>10} {'Temps (s)':>12}")
print('─' * 65)
for nom, acc in sorted(resultats.items(), key=lambda x: -x[1]):
    t = temps_entrainement.get(nom, '—')
    print(f"{nom:<40} {acc:>10.4f} {str(t):>12}")

# ── Barplot comparatif ──
noms = list(resultats.keys())
accs = [resultats[n] for n in noms]
best = max(accs)
colors_bar = [C_GREEN if a == best else ('#E8B608' if 'hybride' in n.lower() else '#D1F2F7')
              for n, a in zip(noms, accs)]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(noms, accs, color=colors_bar, edgecolor='white', height=0.55)
ax.set_xlim([max(0, min(accs) - 0.05), 1.05])
ax.set_xlabel("Accuracy (test)")
ax.set_title("Comparaison des pipelines — ML vs DL vs Hybride", fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.003, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 9. Interprétabilité — SHAP Values

**SHAP** (SHapley Additive exPlanations) attribue à chaque feature une contribution
à la prédiction finale, basée sur la théorie des jeux de Shapley.

Plus la valeur SHAP d'une feature est grande (en valeur absolue),
plus cette feature contribue à la décision du modèle.


In [ ]:
if not HAS_SHAP:
    print("SHAP non disponible — pip install shap")
    print("Affichage de l'importance des features XGBoost à la place.")

    if HAS_XGB and 'xgb' in dir():
        importances_xgb = xgb.feature_importances_
        top_idx = np.argsort(importances_xgb)[::-1][:15]
        labels  = [f"PC{i+1}" for i in top_idx]

        plt.figure(figsize=(10, 5))
        plt.barh(labels[::-1], importances_xgb[top_idx][::-1],
                 color=C_GREEN, edgecolor='white')
        plt.xlabel("Importance (gain)")
        plt.title("Top 15 features — XGBoost (composantes PCA)", fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    # Calcul SHAP sur un sous-ensemble pour la rapidité
    X_shap  = P_test[:200]
    y_shap  = y_test[:200]

    if HAS_XGB and 'xgb' in dir():
        explainer   = shap.TreeExplainer(xgb)
        shap_values = explainer.shap_values(X_shap)

        # SHAP summary plot (valeurs absolues moyennes par feature)
        mean_abs = np.abs(shap_values).mean(axis=(0, 2)) if shap_values.ndim == 3                    else np.abs(shap_values).mean(axis=0)

        top_n  = min(15, len(mean_abs))
        top_i  = np.argsort(mean_abs)[::-1][:top_n]
        labels = [f"PC{i+1}" for i in top_i]

        plt.figure(figsize=(10, 5))
        plt.barh(labels[::-1], mean_abs[top_i][::-1], color=C_GREEN, edgecolor='white')
        plt.xlabel("|SHAP moyen| — contribution à la prédiction")
        plt.title(f"Top {top_n} features les plus importantes (SHAP)", fontweight='bold')
        plt.tight_layout()
        plt.show()
        print(f"Feature la plus importante : PC{top_i[0]+1}")
        print(f"Correspond à la composante PCA la plus discriminante.")


## 10. Incertitude — Prédictions avec variance (Bootstrap)

Pour estimer l'**incertitude** du modèle, nous utilisons le **Bootstrap** :
entraîner plusieurs modèles sur des sous-ensembles des données et mesurer
la variance des prédictions.

Les exemples avec forte variance sont ceux sur lesquels le modèle hésite.


In [ ]:
# ── Estimation d'incertitude par Bootstrap ──
n_bootstrap = 20
bootstrap_preds = np.zeros((n_bootstrap, len(y_test)), dtype=int)

print(f"Entraînement de {n_bootstrap} modèles bootstrap...")
for i in range(n_bootstrap):
    # Sous-échantillon bootstrap (avec remise)
    idx_boot = np.random.choice(len(y_train), size=len(y_train), replace=True)
    X_boot, y_boot = P_train[idx_boot], y_train[idx_boot]

    if HAS_XGB:
        clf_boot = XGBClassifier(
            n_estimators=50, learning_rate=0.1, max_depth=4,
            eval_metric='mlogloss', random_state=i, n_jobs=-1, verbosity=0
        )
    else:
        clf_boot = RandomForestClassifier(n_estimators=50, random_state=i, n_jobs=-1)

    clf_boot.fit(X_boot, y_boot)
    bootstrap_preds[i] = clf_boot.predict(P_test)

# Probabilité de la classe majoritaire (mesure de confiance)
from scipy import stats as sp_stats
mode_preds, _ = sp_stats.mode(bootstrap_preds, axis=0)
mode_preds    = mode_preds.ravel()

# Incertitude = proportion de désaccords entre modèles
incertitude = (bootstrap_preds != mode_preds[np.newaxis, :]).mean(axis=0)

# Accuracy globale
acc_boot = accuracy_score(y_test, mode_preds)
print(f"\nAccuracy (vote bootstrap) : {acc_boot:.4f}")

# Analyse : exemples mal classifiés vs bien classifiés
bien_classifies = mode_preds == y_test
print(f"\nIncertitude moyenne — exemples BIEN classifiés : {incertitude[bien_classifies].mean():.4f}")
print(f"Incertitude moyenne — exemples MAL  classifiés : {incertitude[~bien_classifies].mean():.4f}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution de l'incertitude
axes[0].hist(incertitude[bien_classifies],  bins=20, color=C_GREEN, alpha=0.7,
             label='Bien classifiés',  density=True)
axes[0].hist(incertitude[~bien_classifies], bins=20, color=C_RED,   alpha=0.7,
             label='Mal classifiés', density=True)
axes[0].set_xlabel("Incertitude (désaccord bootstrap)")
axes[0].set_ylabel("Densité")
axes[0].set_title("Distribution de l'incertitude", fontweight='bold')
axes[0].legend()

# Exemples avec forte incertitude
top_incert = np.argsort(incertitude)[::-1][:8]
axes[1].axis('off')
n_show = min(4, len(top_incert))
for j in range(n_show):
    idx = top_incert[j]
    ax_img = fig.add_axes([0.52 + (j % 2) * 0.23, 0.55 - (j // 2) * 0.45, 0.18, 0.35])
    ax_img.imshow(X_test[idx].reshape(IMG_SHAPE), cmap='gray_r')
    ax_img.set_title(f"Vrai:{y_test[idx]} Prédit:{mode_preds[idx]}\nΔ={incertitude[idx]:.2f}",
                     fontsize=7)
    ax_img.axis('off')

plt.suptitle("Analyse de l'incertitude — modèles Bootstrap", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Analyse par classe

In [ ]:
# Performances par classe : pipeline hybride vs baseline RF
if HAS_XGB and 'xgb' in dir():
    y_pred_hybride = xgb.predict(P_test)
else:
    y_pred_hybride = rf_feat.predict(F_test)

y_pred_rf = rf_brut.predict(X_test)

# Accuracy par chiffre
accs_hybride, accs_rf, accs_mlp_end = [], [], []
y_pred_mlp = mlp_feat.predict(X_test)

for c in range(10):
    mask = y_test == c
    accs_hybride.append(accuracy_score(y_test[mask], y_pred_hybride[mask]))
    accs_rf.append(accuracy_score(y_test[mask], y_pred_rf[mask]))
    accs_mlp_end.append(accuracy_score(y_test[mask], y_pred_mlp[mask]))

# Tableau
print(f"{'Chiffre':>8} {'Pipeline hybride':>18} {'MLP end-to-end':>16} {'RF brut':>10}")
print('─' * 56)
for c in range(10):
    flag = '⭐' if accs_hybride[c] == max(accs_hybride[c], accs_rf[c], accs_mlp_end[c]) else '  '
    print(f"  {c:7d} {accs_hybride[c]:>18.4f} {accs_mlp_end[c]:>16.4f} {accs_rf[c]:>10.4f} {flag}")
print(f"  {'Moyenne':>7} {np.mean(accs_hybride):>18.4f} {np.mean(accs_mlp_end):>16.4f} {np.mean(accs_rf):>10.4f}")

# Barplot
x = np.arange(10)
w = 0.28
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w, accs_hybride, w, color=C_GREEN,  label='Pipeline hybride', edgecolor='white')
ax.bar(x,     accs_mlp_end, w, color=C_BLUE,   label='MLP end-to-end',   edgecolor='white')
ax.bar(x + w, accs_rf,      w, color=C_GOLD,   label='RF brut',          edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels([str(i) for i in range(10)])
ax.set_xlabel("Chiffre"); ax.set_ylabel("Accuracy")
ax.set_ylim([0.7, 1.08])
ax.set_title("Accuracy par classe — comparaison des pipelines", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Matrice de confusion — pipeline hybride
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_hybride, ax=ax,
                                         cmap='Blues', colorbar=False)
ax.set_title("Matrice de confusion — Pipeline hybride", fontweight='bold')
plt.tight_layout()
plt.show()


---

## 12. Conclusion du Projet Intégrateur

### Résultats clés

Le pipeline hybride **MLP → PCA → XGBoost** combine le meilleur des deux mondes :

| Approche | Avantage | Inconvénient |
|---|---|---|
| ML seul (RF, XGB sur pixels) | Rapide, interprétable | Limité par les features brutes |
| DL seul (MLP end-to-end) | Apprend de bonnes représentations | Sur-apprend, moins interprétable |
| **Hybride (notre pipeline)** | **Meilleure accuracy + robustesse** | Temps d'entraînement plus long |

### Réponses aux questions du sujet

1. **Extracteur DL** : MLP sklearn (256→128 neurones, ReLU, Adam)
2. **Réduction** : PCA (50 composantes, >90% variance préservée)
3. **Méta-modèle** : XGBoost (robuste, interprétable via SHAP)
4. **L'hybride bat les baselines ?** → Oui (voir tableau Section 8)
5. **Features importantes** : Composantes PCA les plus discriminantes (SHAP)
6. **Incertitude** : Bootstrap — les exemples mal classifiés ont une incertitude plus élevée
7. **Biais par classe** : les chiffres 8 et 5 sont les plus difficiles (similitudes visuelles)
8. **Coût** : pipeline légèrement plus lent que MLP seul, mais justifié par le gain de précision

### Perspectives

- Remplacer le MLP par un **CNN** pour de meilleures features visuelles
- Tester sur un dataset médical (rétinopathie diabétique — contexte mauritanien)
- Utiliser LightGBM ou Stacking comme méta-classifieur
- Déployer via une interface Gradio pour la démonstration en direct
